In [ ]:
import json
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Geometry import Point3D

import numpy as np

In [ ]:
def build_3d_mol_from_arrays(atom_type_array, bond_adjacent_array, positions_3d, bond_types=None):
    """
    Build a 3D RDKit molecule from atom types, bond adjacency array, and 3D positions.
    
    Parameters:
    - atom_type_array: array of shape (N,) with atomic numbers (1=H, 6=C, 7=N, 8=O, etc.)
    - bond_adjacent_array: array of shape (N, N) with bond type indices (upper triangular)
    - positions_3d: array of shape (N, 3) with 3D coordinates
    - bond_types: list of bond type strings (default: [None, 'SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC'])
    
    Returns:
    - RDKit Mol object with 3D coordinates
    """
    
    if bond_types is None:
        bond_types = [None, 'SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC']
    
    # Create editable molecule
    mol = Chem.EditableMol(Chem.Mol())
    
    # Keep track of original to new atom indices (excluding H atoms)
    atom_idx_mapping = {}
    new_atom_idx = 0
    
    # Add atoms (skip hydrogen atoms with atomic number 1)
    for i, atomic_number in enumerate(atom_type_array):
        if atomic_number == 0 or atomic_number == 1:
            continue  # Skip None (0) or hydrogen atoms (1)
            
        # Create atom using atomic number
        atom = Chem.Atom(int(atomic_number))
        mol.AddAtom(atom)
        atom_idx_mapping[i] = new_atom_idx
        new_atom_idx += 1
    
    # Add bonds
    n_atoms = len(atom_type_array)
    for i in range(n_atoms):
        for j in range(i + 1, n_atoms):
            # Skip if either atom was excluded (hydrogen or invalid)
            if i not in atom_idx_mapping or j not in atom_idx_mapping:
                continue
                
            bond_type_idx = bond_adjacent_array[i][j]
            if bond_type_idx == 0 or bond_type_idx >= len(bond_types):
                continue  # Skip None bonds or invalid indices
                
            bond_type_str = bond_types[bond_type_idx]
            
            # Convert bond type string to RDKit bond type
            if bond_type_str == 'SINGLE':
                bond_type = Chem.BondType.SINGLE
            elif bond_type_str == 'DOUBLE':
                bond_type = Chem.BondType.DOUBLE
            elif bond_type_str == 'TRIPLE':
                bond_type = Chem.BondType.TRIPLE
            elif bond_type_str == 'AROMATIC':
                bond_type = Chem.BondType.AROMATIC
            else:
                continue  # Skip unknown bond types
            
            mol.AddBond(atom_idx_mapping[i], atom_idx_mapping[j], bond_type)
    
    # Convert to molecule
    mol = mol.GetMol()
    
    if mol is None:
        return None
    
    # Add 3D coordinates
    conf = Chem.Conformer(mol.GetNumAtoms())
    for orig_idx, new_idx in atom_idx_mapping.items():
        x, y, z = positions_3d[orig_idx]
        conf.SetAtomPosition(new_idx, Point3D(float(x), float(y), float(z)))
    
    mol.AddConformer(conf)
    
    # Sanitize molecule
    try:
        Chem.SanitizeMol(mol)
    except Exception as e:
        print(f"Warning: Could not sanitize molecule: {e}")
        print("Returning unsanitized molecule.")
    
    return mol


In [ ]:
if __name__ == '__main__':
    with open('output.json', 'r') as f:
        data = json.load(f)

    graph_data = data[0]['x1']
    num_atoms = len(graph_data['atoms'])
    atom_types = np.array(graph_data['atoms']).astype(np.int32)
    atom_positions = np.array(graph_data['positions'])
    bond_adj = 1-np.diag(np.ones(num_atoms, dtype = int))  # 创建全连接矩阵，对角线为0（原子不与自己相连）
    bond_adj = np.triu(bond_adj)  # 取上三角矩阵，避免重复边（无向图转有向图表示）
    bond_edge_index = np.stack(bond_adj.nonzero(), axis = 0)
    bond_adj[bond_edge_index[0], bond_edge_index[1]] = graph_data['bonds']

    mol = build_3d_mol_from_arrays(atom_type_array=atom_types, bond_adjacent_array=bond_adj, positions_3d=atom_positions)